# Project Notebook v2 — Part 1: Data Preparation

Split out of `project_notebook_v2.ipynb` (which stays the single-file version). This part loads the raw CSV, cleans/translates it, builds the cohort and the leakage-safe design matrix, and **saves the prepared objects to `artifacts/`** for the visuals (Part 2) and training (Part 3) notebooks.

In [1]:
import pandas as pd

df = pd.read_csv('mexico_covid19.csv')
print(df["RESULTADO"].unique())
translation_dict = {
    "FECHA_ARCHIVO": "report_date",
    "ID_REGISTRO": "registry_id",
    "ENTIDAD_UM": "medical_unit_state",
    "ENTIDAD_RES": "residence_state",
    "RESULTADO": "result", #RT-PCR test result
    "DELAY": "days_to_confirmation",
    "ENTIDAD_REGISTRO": "registration_state",
    "ENTIDAD": "state_name",
    "ABR_ENT": "state_abbreviation",
    "FECHA_ACTUALIZACION": "update_date",
    "ORIGEN": "origin",
    "SECTOR": "sector",
    "SEXO": "sex",
    "ENTIDAD_NAC": "birth_state",
    "MUNICIPIO_RES": "residence_municipality",
    "TIPO_PACIENTE": "patient_type",
    "FECHA_INGRESO": "admission_date",
    "FECHA_SINTOMAS": "symptoms_onset_date",
    "FECHA_DEF": "death_date",
    "INTUBADO": "intubated",
    "NEUMONIA": "pneumonia",
    "EDAD": "age",
    "NACIONALIDAD": "nationality",
    "EMBARAZO": "pregnant",
    "HABLA_LENGUA_INDIG": "speaks_indigenous_language",
    "DIABETES": "diabetes",
    "EPOC": "COPD",
    "ASMA": "asthma",
    "INMUSUPR": "immunosuppressed",
    "HIPERTENSION": "hypertension",
    "OTRA_COM": "other_comorbidity",
    "CARDIOVASCULAR": "cardiovascular",
    "OBESIDAD": "obesity",
    "RENAL_CRONICA": "chronic_renal_failure",
    "TABAQUISMO": "tobacco_use",
    "OTRO_CASO": "contact_with_other_case",
    "MIGRANTE": "migrant",
    "PAIS_NACIONALIDAD": "nationality_country",
    "PAIS_ORIGEN": "origin_country",
    "UCI": "ICU",
}

df = df.rename(columns=translation_dict)

print(df["registration_state"].unique()) #registrato belongs to catalogo de entidades
print(df["residence_state"].unique())  
print(df["residence_municipality"].unique()) #cant be bothered
df["sector"].unique()
print(df["registry_id"].nunique())
print(len(df))

[2 1]
[25 14  8  9 19 17 27 15  5 28 11 24 31  2 21 32 13  3 26 16 30  1 10 12
 20 29 18  7 22 23  6  4]
[25 14  8 15  9 19 17 27  5 28 11 24 31  2 21 32 13  3 26 16 30  1 10 12
 20 29 18  7 22 23  6  4]
[ 13.  98.  19.  33.  15.  39.  12.   4.  20.   3.  18.  26.   5.  38.
  17.  28.   2.  11.  67.  50.  97.  90. 114.  51. 106.   1.   8.  30.
  69.  16. 124.  37.   7.  10.  35. 193. 120.  14.   6.  46.  66.  23.
  48. 109.  25.  83. 178.  77.  72.  31. 156. 105. 101.  59.  27.  57.
  53.  93.  24.  99.  44.  41.  70.  81. 104.  22.  56.  54.   9. 121.
  21. 413. 118.  61. 102. 350.  32.  43. 169. 206.  76. 399. 119.  71.
 140. 173.  58.  29. 157. 185. 324. 122. 108. 107.  73. 117.  40. 184.
  47.  60.  87.  63.  42.  68.  36. 112.  96.  65. 254. 138.  nan 136.
  55.  85. 131.  88. 483.  74. 141. 549.  34.  45. 191.  52. 385. 334.
 409.  49.  64. 474.  75. 196.  80. 100. 133.  89. 181. 553. 125. 142.
 126.  78.  86. 129. 390.  84.  94. 214. 174. 110. 318.  82. 111. 154.
  62. 227. 113.

In [2]:
# we dont translate municipios we dont think there is anything usefull there

# YES/NO catalog: 1=Yes, 2=No, 97=N/A, 98=Ignored, 99=Not specified this is from the catalogo si_no
yes_no_map = {1: 'Yes', 2: 'No', 97: 'N/A', 98: 'Ignored', 99: 'Not specified'}
yes_no_cols = [
    'intubated', 'pneumonia', 'pregnant', 'speaks_indigenous_language',
    'diabetes', 'COPD', 'asthma', 'immunosuppressed', 'hypertension',
    'other_comorbidity', 'cardiovascular', 'obesity', 'chronic_renal_failure',
    'tobacco_use', 'contact_with_other_case', 'migrant', 'ICU',
]
for col in yes_no_cols:
    df[col] = df[col].map(yes_no_map)

# ORIGEN catalogo origin
df['origin'] = df['origin'].map({1: 'USMER', 2: 'Outside USMER', 99: 'Not specified'})

# SECTOR catalogo sector
df['sector'] = df['sector'].map({
    1: 'Red Cross', 2: 'DIF', 3: 'State', 4: 'IMSS', 5: 'IMSS-Bienestar',
    6: 'ISSSTE', 7: 'Municipal', 8: 'PEMEX', 9: 'Private',
    10: 'SEDENA (Army)', 11: 'SEMAR (Navy)', 12: 'SSA (Ministry of Health)',
    13: 'University', 99: 'Not specified',
})

# SEXO catalogo sexo
df['sex'] = df['sex'].map({1: 'Female', 2: 'Male', 99: 'Not specified'})

# TIPO_PACIENTE catalogo tipo_paciente
df['patient_type'] = df['patient_type'].map({1: 'Outpatient', 2: 'Hospitalized', 99: 'Not specified'})

# NACIONALIDAD catalogo naciionalidad
df['nationality'] = df['nationality'].map({1: 'Mexican', 2: 'Foreign', 99: 'Not specified'})

# RESULTADO
df['result'] = df['result'].map({
    1: 'Positive', 2: 'Negative'
})

# Mexican state codes (ENTIDAD_UM, ENTIDAD_RES, ENTIDAD_NAC, ENTIDAD_REGISTRO) catalogo de entidades
state_map = {
    1: 'Aguascalientes', 2: 'Baja California', 3: 'Baja California Sur',
    4: 'Campeche', 5: 'Coahuila', 6: 'Colima', 7: 'Chiapas', 8: 'Chihuahua',
    9: 'Ciudad de México', 10: 'Durango', 11: 'Guanajuato', 12: 'Guerrero',
    13: 'Hidalgo', 14: 'Jalisco', 15: 'Estado de México', 16: 'Michoacán',
    17: 'Morelos', 18: 'Nayarit', 19: 'Nuevo León', 20: 'Oaxaca',
    21: 'Puebla', 22: 'Querétaro', 23: 'Quintana Roo', 24: 'San Luis Potosí',
    25: 'Sinaloa', 26: 'Sonora', 27: 'Tabasco', 28: 'Tamaulipas',
    29: 'Tlaxcala', 30: 'Veracruz', 31: 'Yucatán', 32: 'Zacatecas', 36:'Mexico',
    97: 'N/A', 98: 'Ignored', 99: 'Not specified',
}
for col in ['medical_unit_state', 'residence_state', 'birth_state', 'registration_state']:
    df[col] = df[col].map(state_map)

# Date columns — replace death_date sentinel then convert all to datetime
df['death_date'] = df['death_date'].replace('9999-99-99', None)

date_cols = ['report_date', 'update_date', 'admission_date', 'symptoms_onset_date', 'death_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')

df

,id,report_date,registry_id,medical_unit_state,residence_state,result,days_to_confirmation,registration_state,state_name,state_abbreviation,...,other_comorbidity,cardiovascular,obesity,chronic_renal_failure,tobacco_use,contact_with_other_case,migrant,nationality_country,origin_country,ICU
0,9269,2020-04-12,00011f,Sinaloa,Sinaloa,Negative,0,Sinaloa,Sinaloa,SL,...,No,No,Yes,No,No,No,Not specified,MÃ©xico,97,N/A
1,33333,2020-04-12,00014e,Jalisco,Jalisco,Negative,0,Jalisco,Jalisco,JC,...,No,No,Yes,No,Yes,Not specified,Not specified,MÃ©xico,97,No
2,35483,2020-04-12,000153,Chihuahua,Chihuahua,Positive,0,Chihuahua,Chihuahua,CH,...,No,No,No,No,No,Not specified,Not specified,MÃ©xico,97,No
3,7062,2020-04-12,0001b6,Ciudad de México,Estado de México,Positive,0,Ciudad de México,Ciudad de Mexico,DF,...,No,No,Yes,No,No,Not specified,Not specified,MÃ©xico,97,N/A
4,23745,2020-04-12,0001c1,Ciudad de México,Ciudad de México,Negative,0,Ciudad de México,Ciudad de Mexico,DF,...,No,No,No,No,No,Not specified,Not specified,MÃ©xico,97,N/A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263002,7094887,2020-06-03,1e019c,Zacatecas,Zacatecas,Positive,0,Zacatecas,Zacatecas,ZS,...,No,No,No,No,No,Yes,Not specified,MÃ©xico,99,N/A
263003,7053721,2020-06-03,1e2b05,Guerrero,Guerrero,Positive,0,Guerrero,Guerrero,GR,...,No,No,Yes,No,No,Not specified,Not specified,MÃ©xico,99,No
263004,7055429,2020-06-03,1e473f,Oaxaca,Oaxaca,Positive,0,Oaxaca,Oaxaca,OC,...,No,No,No,Yes,No,Not specified,Not specified,MÃ©xico,99,No
263005,7043768,2020-06-03,1e6da1,Hidalgo,Hidalgo,Positive,0,Hidalgo,Hidalgo,HG,...,No,No,No,No,No,No,Not specified,MÃ©xico,99,No


# Section A — Problem Formulation & Dataset Analysis

**Prediction task.** Binary **classification**: using only information available at a patient's *initial presentation* to a sentinel unit, predict whether that patient will be **hospitalized** (`patient_type` = Hospitalized) rather than treated as an outpatient. We restrict to **RT-PCR-positive** patients, so the task reads: *among lab-confirmed COVID-19 patients presenting at a sentinel unit, who gets admitted?*

- **Target:** `patient_type == 'Hospitalized'` (1) vs `Outpatient` (0).
- **Prediction population:** symptomatic, lab-confirmed COVID-19 patients evaluated at the 475 SISVER sentinel units (data spans Apr–Jun 2020).
- **Prediction moment:** first clinical contact / registration.
- **Prediction horizon:** the admission decision for the current episode.
- **Information at prediction time:** demographics, registration metadata, and pre-existing comorbidities. Variables recorded at or after admission/discharge (intubation, ICU, in-hospital pneumonia, death) are treated as leakage — see the leakage audit below.

**Population interpretation & collection bias.** SISVER rationed testing by severity: 100% of severe/SARI patients but only ~10% of mild symptomatic patients were tested, and asymptomatic people essentially never. Confirmed positives are therefore **over-weighted toward severe presentations**, so the hospitalization rate we model is *not* a population-level admission probability for everyone infected — it is the admission rate *within this severity-skewed, testing-selected sample*. The **dataset population** (tested, mostly severe presentations at sentinel units) is narrower than the **target population** we would like to generalize to (all symptomatic COVID-19 patients), and this selection bias limits generalization. We do not attempt to correct it; instead we interpret results conditional on this sampling.

> **[V2 CHANGE]** In this version the cohort is **not** restricted to positive tests by default — all presenting patients are included (see the `INCLUDE_ONLY_POSITIVE` flag in the cohort cell). Set that flag to `True` to restore the original positives-only cohort described above.

In [3]:
# --- Reproducibility ---
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ===== [V2 CHANGED] Cohort is no longer restricted to positive tests by default. =====
# v1 kept ONLY positive RT-PCR patients:
#     cohort = df[df['result'] == 'Positive'].copy()
# v2 includes ALL presenting patients by default. To revert to the original
# positives-only cohort, set the flag below to True (or delete this block and
# restore the single commented line above).
INCLUDE_ONLY_POSITIVE = False
if INCLUDE_ONLY_POSITIVE:
    cohort = df[df['result'] == 'Positive'].copy()
    print(f"[cohort] positives only: {len(cohort):,} records")
else:
    cohort = df.copy()
    print(f"[cohort] all patients (positivity filter OFF): {len(cohort):,} records")
# ===== end [V2 CHANGED] =====

# --- De-duplicate repeated patient records ---
# The source is a stack of daily archive snapshots (report_date); the same
# registry_id can appear on several days. Keep the latest snapshot per patient.


# --- Target: hospitalized (1) vs outpatient (0) ---
cohort = cohort[cohort['patient_type'].isin(['Outpatient', 'Hospitalized'])].copy()
cohort['target'] = (cohort['patient_type'] == 'Hospitalized').astype(int)

print("\nClass distribution (0 = Outpatient, 1 = Hospitalized):")
print(cohort['target'].value_counts().sort_index())
print(f"Hospitalization prevalence: {cohort['target'].mean():.3f}")


[cohort] all patients (positivity filter OFF): 263,007 records

Class distribution (0 = Outpatient, 1 = Hospitalized):
target
0    200838
1     62169
Name: count, dtype: int64
Hospitalization prevalence: 0.236


### Feature selection (leakage-aware)

We keep only variables plausibly known at first contact: **age**, **sex**, registration metadata (`sector`, `origin`/USMER, `residence_state`, `medical_unit_state`, `nationality`, `contact_with_other_case`), and **pre-existing conditions**. Rather than statistically imputing missing values, we handle each case explicitly:

- **`migrant`** — dropped entirely (99.7% *Not specified*, no signal).
- **`pregnant`** and **`speaks_indigenous_language`** — *not specified* is set to **No (0)** (a fixed domain rule, not learned from data).
- **`contact_with_other_case`** — *not specified* is kept as an explicit third category (`Unknown`).
- **the comorbidity flags (and `age`)** — records that are not filled in are **dropped** (only a small fraction; see the drop report below).

Binary Yes/No fields are encoded `Yes→1`, `No→0`, and categoricals are one-hot encoded. Because every remaining missing value is resolved here, the model pipeline needs **no imputation step**. Variables recorded downstream of the admission decision — `intubated`, `ICU`, `pneumonia`, `death_date`, `days_to_confirmation`, and the various dates — are excluded, because they are not known at the time the admission decision is made.

> **[V2 CHANGES]** `speaks_indigenous_language` is dropped from the feature set, and an engineered feature **`N_COMORBIDITIES`** (count of chronic conditions) is added to the numeric block.

In [4]:
numeric_features = ['age']
comorbidity_cols = [
    'diabetes', 'COPD', 'asthma', 'immunosuppressed', 'hypertension',
    'other_comorbidity', 'cardiovascular', 'obesity', 'chronic_renal_failure',
    'tobacco_use',
]
# ===== [V2 CHANGED] 'speaks_indigenous_language' dropped from the feature set. =====
# v1 was: binary_features = comorbidity_cols + ['pregnant', 'speaks_indigenous_language']
binary_features = comorbidity_cols + ['pregnant', 'treated_in_residence_state']
# ===== end [V2 CHANGED] =====

# Categorical (one-hot). `contact_with_other_case` keeps 'not specified' as a third category;
# `migrant` is dropped entirely (99.7% 'Not specified', no signal).
# ===== [V3 ADDED] 'medical_unit_state' (state of the treating medical unit) added as a feature. =====
# Known at presentation (it is where the patient shows up), one-hot encoded like residence_state.
categorical_features = ['sex', 'sector', 'origin', 'residence_state', 'medical_unit_state',
                        'nationality', 'contact_with_other_case']
# ===== end [V3 ADDED] =====

# ===== [V3 ADDED] Engineered geo feature: treated in home state? =====
# 'Yes' if the treating medical unit is in the patient's residence state, else 'No'.
# Captures referral / out-of-state care and local admission patterns; known at presentation.
cohort['treated_in_residence_state'] = np.where(
    cohort['medical_unit_state'] == cohort['residence_state'], 'Yes', 'No')
# ===== end [V3 ADDED] =====

# Base columns pulled from the cohort (engineered N_COMORBIDITIES is added after cleaning).
base_cols = numeric_features + binary_features + categorical_features

# Modelling frame: leakage-safe features + target.
# Also carry a descriptive-only 'died' flag (from death_date). It is NOT added to
# feature_cols, so it never enters the design matrix X (that would be leakage).
data = cohort[base_cols + ['target']].copy()
data['died'] = cohort['death_date'].notna().astype(int)
data = data.reset_index(drop=True)

# Encode binary Yes/No -> 1/0
binary_map = {'Yes': 1, 'No': 0}
for col in binary_features:
    data[col] = data[col].map(binary_map)

# `pregnant`: not-specified -> No (0). A fixed domain rule, not learned from data.
# ===== [V2 CHANGED] 'speaks_indigenous_language' removed from this fill loop (feature dropped). =====
for col in ['pregnant']:
    data[col] = data[col].fillna(0)
# ===== end [V2 CHANGED] =====

# `contact_with_other_case`: keep 'not specified' as an explicit third category ('Unknown').
data['contact_with_other_case'] = data['contact_with_other_case'].where(
    data['contact_with_other_case'].isin(['Yes', 'No']), 'Unknown')

# The remaining features (comorbidities + age): no imputation — drop rows not filled in.
drop_subset = comorbidity_cols + numeric_features
rows_before = len(data)
missing_per_col = data[drop_subset].isna().sum()
data = data.dropna(subset=drop_subset).reset_index(drop=True)
rows_after = len(data)

# ===== [V2 ADDED] Engineered feature: N_COMORBIDITIES (count of chronic conditions). =====
# Comorbidity flags are clean 0/1 here (rows with any missing flag were dropped above),
# so the row-wise sum is well defined. Added to the NUMERIC block so it is scaled with age.
data['N_COMORBIDITIES'] = data[comorbidity_cols].sum(axis=1).astype(int)
numeric_features = numeric_features + ['N_COMORBIDITIES']
# ===== end [V2 ADDED] =====

# Final feature list (now includes the engineered feature, excludes speaks_indigenous_language)
feature_cols = numeric_features + binary_features + categorical_features

X = data[feature_cols]
y = data['target']

print(f"Design matrix: {X.shape[0]:,} rows x {X.shape[1]} features")
print(f"Numeric: {numeric_features} | Binary: {len(binary_features)} | Categorical: {categorical_features}")
print(f"Remaining missing values in X: {int(X.isna().sum().sum())}")


Design matrix: 260,933 rows x 21 features
Numeric: ['age', 'N_COMORBIDITIES'] | Binary: 12 | Categorical: ['sex', 'sector', 'origin', 'residence_state', 'medical_unit_state', 'nationality', 'contact_with_other_case']
Remaining missing values in X: 0


In [5]:
# How much data did the "drop rows with unfilled comorbidities" rule remove?
dropped = rows_before - rows_after
print("=== Rows dropped for unfilled comorbidity / age values ===")
print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Dropped     : {dropped:,}  ({100 * dropped / rows_before:.2f}%)")

print("\n'Not specified' (missing) count per dropped-on column, before dropping:")
print(missing_per_col[missing_per_col > 0].sort_values(ascending=False))

print(f"\nHospitalization prevalence — before: {cohort['target'].mean():.3f} | "
      f"after: {y.mean():.3f}   (a small change means the drop did not skew the target)")

=== Rows dropped for unfilled comorbidity / age values ===
Rows before : 263,007
Rows after  : 260,933
Dropped     : 2,074  (0.79%)

'Not specified' (missing) count per dropped-on column, before dropping:
other_comorbidity        1344
immunosuppressed         1038
diabetes                 1011
tobacco_use               982
obesity                   962
cardiovascular            961
hypertension              938
chronic_renal_failure     937
COPD                      931
asthma                    923
dtype: int64

Hospitalization prevalence — before: 0.236 | after: 0.234   (a small change means the drop did not skew the target)


In [6]:
# Treated in home state? Flag whether the medical unit's state matches the patient's residence state.
df['treated_in_residence_state'] = np.where(
    df['medical_unit_state'] == df['residence_state'], 'Yes', 'No')

counts = df['treated_in_residence_state'].value_counts()
print("medical_unit_state == residence_state?")
print(counts)
print(f"\nYes: {counts.get('Yes', 0):,}  ({counts.get('Yes', 0) / len(df):.1%})")

print(f"No : {counts.get('No', 0):,}  ({counts.get('No', 0) / len(df):.1%})")

medical_unit_state == residence_state?
treated_in_residence_state
Yes    243457
No      19550
Name: count, dtype: int64

Yes: 243,457  (92.6%)
No : 19,550  (7.4%)


### Leakage audit

Prediction moment = initial presentation, so anything recorded at/after admission or discharge is excluded from `X`.

| Variable | Available at prediction time? | Included? | Reason |
|---|---|---|---|
| age, sex | Yes | Yes | Baseline demographics |
| comorbidities (diabetes, COPD, asthma, immunosuppressed, hypertension, cardiovascular, obesity, chronic renal failure, tobacco use, other), pregnant | Yes | Yes | Pre-existing, known at presentation |
| sector, origin (USMER), residence_state, nationality, speaks_indigenous_language, contact_with_other_case | Yes | Yes | Recorded at registration |
| migrant | Yes, but 99.7% `Not specified` | No | Dropped for lack of signal (a data-quality choice, not leakage) |
| **pneumonia** | No — recorded as an in-hospital diagnosis | No | Not known at the time of the admission decision |
| **intubated, ICU** | No | No | Occur only after admission |
| death_date | No | No | Final outcome |
| days_to_confirmation (DELAY) | No | No | Post-presentation lab timing |
| admission_date, report_date, update_date | No | No | Post-presentation / archival |
| result | Defines cohort | No (as feature) | Used only to select positive patients |
| **patient_type** | — | **TARGET** | Hospitalized = 1, Outpatient = 0 |

> **[V2]** `speaks_indigenous_language` removed from features; `N_COMORBIDITIES` (engineered count, known at presentation) added; cohort positivity filter now optional (`INCLUDE_ONLY_POSITIVE`).

In [7]:
# --- Persist prepared objects for the visuals & training notebooks ---
import os

os.makedirs('artifacts', exist_ok=True)
df.to_pickle('artifacts/df_clean.pkl')       # cleaned full dataset (EDA time series uses it)
cohort.to_pickle('artifacts/cohort.pkl')     # modelling cohort (with target)
data.to_pickle('artifacts/data.pkl')         # modelling frame (features + target + died flag)

import json
with open('artifacts/feature_lists.json', 'w') as f:
    json.dump({
        'numeric_features': numeric_features,
        'binary_features': binary_features,
        'categorical_features': categorical_features,
        'comorbidity_cols': comorbidity_cols,
        'feature_cols': feature_cols,
    }, f, indent=2)

print("Saved to artifacts/: df_clean.pkl, cohort.pkl, data.pkl, feature_lists.json")

Saved to artifacts/: df_clean.pkl, cohort.pkl, data.pkl, feature_lists.json
